<a href="https://colab.research.google.com/github/hodl17/MN5162-Natural-Language-Understanding/blob/main/MN5162_Assignment1_24346152.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Eoin Devlin -
24346152

Github repo: https://github.com/hodl17/MN5162-Natural-Language-Understanding

In [186]:
!pip install -q transformers==4.51.3
!pip install huggingface_hub
!pip install accelerate
!pip install -q python-docx
!pip install -q diffusers

In [187]:
from huggingface_hub import login
login()

In [188]:
import pandas as pd
import re
import os
from transformers import pipeline

In [189]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [190]:
output_path = "/content/drive/MyDrive/MN5162/"

# Generate Letters

In [191]:
letter_gen_pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-3.5-mini-instruct",
    )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [192]:
def extract_letter(text, customer):
    start = text.find("Dear")
    if start == -1:
        return None

    end = text.find(customer, start)
    if end == -1:
        return None

    return text[start:end + len(customer)]

sentiment_levels = {
    # level: category, tone, request, customer
    1: ("praise", "neutral", "None", "Derek"),
    2: ("praise", "polite", "Keep up the good work", "Aine"),
    3: ("praise", "delighted", "Free samples", "Patrick"),
    4: ("praise", "ecstatic", "Free samples", "Siobhan"),

    5: ("complaint", "firm and serious", "Exchange or refund. Explanation on how it happened.", "Mark"),
    6: ("complaint", "strongly dissatisfied", "Exchange or refund. Explanation on how it happened.", "Marion"),
    7: ("complaint", "very angry", "Refund and further compensation.", "Hugh"),
    8: ("complaint", "Unhinged to the point of violence", "A refund or failing that a set of wings and an operation to transplant them", "Karen"),
}

product = "energy drink"
purpose = {
    "praise": "Really like the taste and energy boost",
    "complaint": "Did not cause me to grow wings as advertised"
}
company = "Blue Cow"


output = {}
for level, details in sentiment_levels.items():
    cat, tone, request, customer = details

    prompt = f"""
    You are writing exactly one formal {cat} letter.

    Do not write instructions.
    Do not write bullet points.
    Do not write multiple letters.
    Do not add headings or HTML tags.

    Write a {cat} letter with this tone: {tone}.

    Product: {product}
    Issue: {purpose[cat]}
    Requested resolution: {request}
    Company name: {company}

    The output must be a complete letter with:
    - greeting (1 line)
    - explanation of the reason for the letter
    - request for resolution
    - professional closing

    Structure:
    - Greeting (1 line)
    - Body (2 paragraphs, max 6 sentences each)
    - Closing sentence (1 sentence)
    - Sign-off ("Yours sincerely," + name)

    Rules:
    - Keep sentences concise
    - Do NOT exceed the structure
    - Always include the sign-off at the end

    You must close the letter by signing off with the customer name: {customer}

    After the closing end the output and do not start another letter.
    """

    result = letter_gen_pipe(
        prompt,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.3,
        return_full_text=False,
        max_length=None
    )
    output[customer] = result[0]["generated_text"]

In [193]:
letters = {x: extract_letter(y,x) for x,y in output.items()}

In [194]:
# write letters to drive
def clean_filename(name):
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name)

for name, letter in letters.items():
    filename = os.path.join(output_path+"letters", f"{clean_filename(name)}.txt")

    with open(filename, "w", encoding="utf-8") as f:
        f.write(letter.strip())

## Justify model usage
I use the microsoft/Phi-3.5-mini-instruct text generation model. A main factor is that its training data includes "high quality chat format supervised data covering various topics to reflect human preferences on different aspects such as instruct-following, truthfulness, honesty and helpfulness." This suggested a good fit for following instructions and generating human-sounding letters.

I also experimented with Qwen/Qwen2.5-3B-Instruct and google/gemma-2b but found that they hit edge cases for the letter generation, sometimes listing the steps required to write the letter rather than actually writing a letter.

For data processing I wrote an extract_letter() function which returns the first complete letter in the output. If the generated letter is shorter than max_new_tokens the model often starts a new letter. So I truncate the output to just the first generated letter.

# Sentiment Analysis

In [195]:
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
    )

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [196]:
names = list(letters.keys())
letters_list = list(letters.values())

In [197]:
results = sentiment_analyzer(letters_list)

df = pd.DataFrame({
    "name": names,
    "sentiment": [r["label"] for r in results],
    "score": [r["score"] for r in results]
})
df

,name,sentiment,score
0,Derek,positive,0.943304
1,Aine,positive,0.978928
2,Patrick,positive,0.978617
3,Siobhan,positive,0.982127
4,Mark,negative,0.742994
5,Marion,negative,0.717098
6,Hugh,negative,0.896436
7,Karen,negative,0.805090


## Justify Model Usage
I use cardiffnlp/twitter-roberta-base-sentiment-latest which is trained on Twitter data. I also experimented with distilbert-base-uncased-finetuned-sst-2-english but found it to often misclassify the letters. I am not sure why, perhaps there is some bias in the underlying data as distiilbert is trained on the Standford movie review corpus.

# Summarization

In [198]:
summarizer = pipeline(
    "summarization",
    # model="facebook/bart-large-cnn",
    )

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


In [199]:
combined_text_neg = "\n\n".join(
    f"Letter from {name}:\n{letter}"
    for name, letter in letters.items()
    if df[df["name"]==name]["sentiment"].to_list()[0] == "negative"
)

combined_text_pos = "\n\n".join(
    f"Letter from {name}:\n{letter}"
    for name, letter in letters.items()
    if df[df["name"]==name]["sentiment"].to_list()[0] == "positive"
)

In [200]:
summary_results = summarizer(
    letters_list,
    max_length=80,
    min_length=20,
    do_sample=False,
    num_beams=4,
    length_penalty=1.0,
    no_repeat_ngram_size=3,
    batch_size=4,
    truncation=True,
)

In [201]:
df["summary"] = [r["summary_text"] for r in summary_results]
df

,name,sentiment,score,summary
0,Derek,positive,0.943304,"The taste is refreshing, and I have noticed a..."
1,Aine,positive,0.978928,"The taste is delightful, and the energy boost..."
2,Patrick,positive,0.978617,The refreshing flavor and invigorating effect...
3,Siobhan,positive,0.982127,The invigorating taste and the remarkable ene...
4,Mark,negative,0.742994,Blue Cow energy drink is not only misleading ...
5,Marion,negative,0.717098,The product was marketed with the bold claim ...
6,Hugh,negative,0.896436,The experience has been nothing short of a sc...
7,Karen,negative,0.805090,"The promise of gaining superhuman abilities, ..."


In [202]:
summary_results_pos = summarizer(
    combined_text_pos,
    max_length=80,
    min_length=20,
    do_sample=False,
    num_beams=4,
    length_penalty=1.0,
    no_repeat_ngram_size=3,
    batch_size=4,
    truncation=True,
)
summary_results_pos

[{'summary_text': ' Letter from Derek: "I am pleased with the product and its performance" Letter from Aine: "The taste is delightful, and the energy boost it provides has significantly enhanced my daily routine"'}]

In [203]:
summary_results_neg = summarizer(
    combined_text_neg,
    max_length=80,
    min_length=20,
    do_sample=False,
    num_beams=4,
    length_penalty=1.0,
    no_repeat_ngram_size=3,
    batch_size=4,
    truncation=True,
)
summary_results_neg

[{'summary_text': ' The claim that this energy drink could potentially enable one to grow wings is not only misleading but also unrealistic . I am requesting a full refund or an exchange for a product that meets the standards of honesty and realism .'}]

## Justify model usage
I use the default summarizer model which is sshleifer/distilbart-cnn-12-6. It produced the most sensible and concise summaries compared to other models I experimented with like facebook/bart-large-cnn. I also wanted to try some google models like google/flan-t5-base but I am still waiting on access to the model on hugging face.

# Question Answering

In [204]:
qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
    )

Device set to use cuda:0


In [205]:
question = "What remediation action does the customer request the company takes?"

# create batch input
qa_inputs = [
    {"question": question, "context": letter}
    for letter in letters_list
]

# run in batch
results = qa_model(qa_inputs, batch_size=4)

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


In [206]:
df["request"] = [r["answer"] for r in results]
df

,name,sentiment,score,summary,request
0,Derek,positive,0.943304,"The taste is refreshing, and I have noticed a...",not requesting any specific action
1,Aine,positive,0.978928,"The taste is delightful, and the energy boost...",\nAine
2,Patrick,positive,0.978617,The refreshing flavor and invigorating effect...,free samples
3,Siobhan,positive,0.982127,The invigorating taste and the remarkable ene...,free samples
4,Mark,negative,0.742994,Blue Cow energy drink is not only misleading ...,a full refund
5,Marion,negative,0.717098,The product was marketed with the bold claim ...,a full refund
6,Hugh,negative,0.896436,The experience has been nothing short of a sc...,full refund of the purchase price
7,Karen,negative,0.805090,"The promise of gaining superhuman abilities, ...",a miraculous solution that includes a set of w...


## Justify model usage

I use deepset/roberta-base-squad2 for question answering. The hardest part of this was prompting with a precise enough question to get sensible answers. You can see in the more neutral of the letters on the positive side (i.e. those where no meaningful request is made) that the model doesn't find a request.

# Most extreme

In [207]:
most_positive = df[df["sentiment"]=="positive"].sort_values("score", ascending=False).head(1)
most_negative = df[df["sentiment"]=="negative"].sort_values("score", ascending=False).head(1)

In [208]:
print(f"""
  The most positive letter is from {most_positive["name"].to_list()[0]}.
  Summary: {most_positive["summary"].to_list()[0]}
  Request: {most_positive["request"].to_list()[0]}
""")

print(f"""
  The most negative letter is from {most_negative["name"].to_list()[0]}.
  Summary: {most_negative["summary"].to_list()[0]}
  Request: {most_negative["request"].to_list()[0]}
""")


  The most positive letter is from Siobhan.
  Summary:  The invigorating taste and the remarkable energy boost it provides have significantly enhanced my daily routine . I am thrilled with the product and would like to request a few free samples to share with friends .
  Request: free samples


  The most negative letter is from Hugh.
  Summary:  The experience has been nothing short of a scam, as the drink did not deliver on its outrageous claims . The deception perpetrated by Blue Cow has left me with a deep sense of betrayal, and it is only fair that I be compensated for the time and resources wasted .
  Request: full refund of the purchase price



## Response Generation

In [209]:
generator = pipeline(
    "text-generation",
    model="microsoft/Phi-3.5-mini-instruct",
    device_map="auto",
    torch_dtype="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [227]:
customer_letter = letters[most_negative["name"].to_list()[0]]

prompt = f"""
You are a customer service representative.

Write a professional response to the customer's complaint letter.

Rules:
- Be polite and empathetic
- Acknowledge the problem
- Apologize briefly
- State what action the company will take
- Keep it concise
- Write only the response letter
- Do not include notes or explanations

End the letter by signing off with: Sincerely, Blue Cow Customer Service Team

Customer letter:
{customer_letter}

Response:
"""

result = generator(
    prompt,
    max_new_tokens=250,
    do_sample=True,
    temperature=0.6,
    return_full_text=False
)

print(result[0]["generated_text"])

Dear Mr. Hugh,

Thank you for reaching out to us and expressing your concerns regarding the Blue Cow energy drink. We sincerely apologize for any disappointment or inconvenience our product may have caused you.

We understand how disheartening it can be when expectations are not met, and we regret that the advertising for our product did not live up to your hopes of achieving the impossible. We value our customers' trust, and we take your feedback seriously.

To address your concerns, we would like to offer you a full refund for your purchase. Additionally, we will investigate the matter further to ensure that our advertising is clear and accurate, preventing similar situations from occurring in the future.

We appreciate your patience and understanding as we work to resolve this issue. Our customer service team will contact you shortly to guide you through the refund process.

Once again, we apologize for any distress caused and thank you for bringing this matter to our attention. We 

In [228]:
response_neg = extract_letter(result[0]["generated_text"],"Blue Cow Customer Service Team")

In [225]:
customer_letter = letters[most_positive["name"].to_list()[0]]

prompt = f"""
You are a customer service representative.

Write a professional response to the customer's praise letter.

Rules:
- Be polite and empathetic
- Acknowledge their appreciation
- State what action the company will take
- Keep it concise
- Write only the response letter
- Do not include notes or explanations

End the letter by signing off with: Sincerely, Blue Cow Customer Service Team

Customer letter:
{customer_letter}

Response:
"""

result = generator(
    prompt,
    max_new_tokens=250,
    do_sample=True,
    temperature=0.6,
    return_full_text=False
)

print(result[0]["generated_text"])


Dear Siobhan,

Thank you for your kind words and for choosing Blue Cow for your energy needs. We are delighted to hear that you've found our energy drink both invigorating and beneficial to your daily routine. Your feedback is greatly appreciated and helps us to continue improving our products.

We are pleased to offer you a few complimentary samples for you to share with friends. We trust that they will enjoy the taste and quality as much as you have. We value your loyalty and look forward to your continued support for Blue Cow.

Sincerely,
Blue Cow Customer Service Team





In [226]:
response_pos = extract_letter(result[0]["generated_text"],"Blue Cow Customer Service Team")

# Create Logo

In [214]:
# from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image
# import torch

# pipe = AutoPipelineForText2Image.from_pretrained(
#     "stabilityai/stable-diffusion-xl-base-1.0",
#     torch_dtype=torch.float16,
#     variant="fp16"
# ).to("cuda")

# image = pipe("Blue cow with wings. Corporate logo.").images[0]
# image.save(output_path+"logo.png")

# Create Report

In [215]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

In [216]:
import numpy as np
import re
import matplotlib.pyplot as plt
plt.style.use("ggplot")

In [217]:
GG_BLUE = "4C72B0"
GG_RED = "C44E52"
GG_GREEN = "55A868"
GG_LIGHT = "EAEAF2"
GG_GRID = "D8D8E0"
GG_TEXT = "2F2F2F"
GG_BOX = "F7F8FC"
WHITE = "FFFFFF"

def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)

def set_cell_border(cell, color=GG_GRID, size="6"):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = tcPr.first_child_found_in("w:tcBorders")
    if tcBorders is None:
        tcBorders = OxmlElement("w:tcBorders")
        tcPr.append(tcBorders)

    for edge in ("top", "left", "bottom", "right"):
        tag = f"w:{edge}"
        element = tcBorders.find(qn(tag))
        if element is None:
            element = OxmlElement(tag)
            tcBorders.append(element)
        element.set(qn("w:val"), "single")
        element.set(qn("w:sz"), size)
        element.set(qn("w:color"), color)

def style_paragraph(paragraph, size=11, bold=False, color=GG_TEXT, space_after=6, space_before=0):
    fmt = paragraph.paragraph_format
    fmt.space_after = Pt(space_after)
    fmt.space_before = Pt(space_before)
    fmt.keep_together = True
    fmt.keep_with_next = False

    for run in paragraph.runs:
        run.font.name = "Aptos"
        run.font.size = Pt(size)
        run.bold = bold
        run.font.color.rgb = RGBColor.from_string(color)

def add_logo_to_header(doc, logo_path, width_inches=0.9):
    section = doc.sections[0]
    header = section.header
    p = header.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)

    # Clear default header content
    p.clear()

    run = p.add_run()
    run.add_picture(logo_path, width=Inches(width_inches))

def add_report_title(doc, title_text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_after = Pt(6)
    run = p.add_run(title_text)
    run.bold = True
    run.font.name = "Aptos"
    run.font.size = Pt(20)
    run.font.color.rgb = RGBColor.from_string(GG_BLUE)

def add_subtitle(doc, text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(text)
    run.font.name = "Aptos"
    run.font.size = Pt(10)
    run.font.color.rgb = RGBColor.from_string("666666")
    p.paragraph_format.space_after = Pt(12)

def add_heading_custom(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_before = Pt(10 if level == 1 else 6)
    p.paragraph_format.space_after = Pt(4)

    run = p.add_run(text)
    run.bold = True
    run.font.name = "Aptos"
    run.font.color.rgb = RGBColor.from_string(GG_BLUE if level <= 2 else GG_TEXT)
    run.font.size = Pt(14 if level == 1 else 12 if level == 2 else 11)

def add_body_text(doc, text):
    p = doc.add_paragraph(text)
    style_paragraph(p, size=10.5, color=GG_TEXT, space_after=6)
    return p

def add_caption(doc, text):
    p = doc.add_paragraph(text)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    style_paragraph(p, size=9, color="666666", space_after=8)

def add_df_to_doc_styled(doc, df, title=None):
    if title:
        add_heading_custom(doc, title, level=3)

    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False

    # Header
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).title()
        set_cell_shading(hdr[i], GG_BLUE)
        set_cell_border(hdr[i], color=GG_BLUE, size="8")
        for p in hdr[i].paragraphs:
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            for r in p.runs:
                r.font.name = "Aptos"
                r.font.size = Pt(10)
                r.bold = True
                r.font.color.rgb = RGBColor.from_string(WHITE)

    # Body
    for row_idx, (_, row) in enumerate(df.iterrows(), start=1):
        cells = table.add_row().cells
        for col_idx, value in enumerate(row):
            if isinstance(value, float):
                value = f"{value:.2f}"

            cells[col_idx].text = str(value)
            set_cell_border(cells[col_idx], color=GG_GRID, size="4")

            if row_idx % 2 == 0:
                set_cell_shading(cells[col_idx], GG_LIGHT)

            for p in cells[col_idx].paragraphs:
                p.paragraph_format.space_before = Pt(0)
                p.paragraph_format.space_after = Pt(0)
                p.paragraph_format.keep_together = True
                for r in p.runs:
                    r.font.name = "Aptos"
                    r.font.size = Pt(9.5)
                    r.font.color.rgb = RGBColor.from_string(GG_TEXT)

    doc.add_paragraph().paragraph_format.space_after = Pt(4)
    return table

def add_textbox_block(doc, title, body, fill=GG_BOX, border=GG_BLUE):
    table = doc.add_table(rows=1, cols=1)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    cell = table.cell(0, 0)

    set_cell_shading(cell, fill)
    set_cell_border(cell, color=border, size="8")
    cell.text = ""

    p1 = cell.add_paragraph()
    p1.paragraph_format.space_before = Pt(0)
    p1.paragraph_format.space_after = Pt(3)
    r1 = p1.add_run(title)
    r1.bold = True
    r1.font.name = "Aptos"
    r1.font.size = Pt(10)
    r1.font.color.rgb = RGBColor.from_string(GG_BLUE)

    p2 = cell.add_paragraph()
    p2.paragraph_format.space_before = Pt(0)
    p2.paragraph_format.space_after = Pt(0)
    r2 = p2.add_run(body)
    r2.font.name = "Aptos"
    r2.font.size = Pt(10)
    r2.font.color.rgb = RGBColor.from_string(GG_TEXT)

    doc.add_paragraph().paragraph_format.space_after = Pt(4)

In [218]:
df["polarity"] = df["score"].round(2)
df.loc[df["sentiment"] == "negative", "polarity"] *= -1

In [219]:
df

,name,sentiment,score,summary,request,polarity
0,Derek,positive,0.943304,"The taste is refreshing, and I have noticed a...",not requesting any specific action,0.94
1,Aine,positive,0.978928,"The taste is delightful, and the energy boost...",\nAine,0.98
2,Patrick,positive,0.978617,The refreshing flavor and invigorating effect...,free samples,0.98
3,Siobhan,positive,0.982127,The invigorating taste and the remarkable ene...,free samples,0.98
4,Mark,negative,0.742994,Blue Cow energy drink is not only misleading ...,a full refund,-0.74
5,Marion,negative,0.717098,The product was marketed with the bold claim ...,a full refund,-0.72
6,Hugh,negative,0.896436,The experience has been nothing short of a sc...,full refund of the purchase price,-0.90
7,Karen,negative,0.805090,"The promise of gaining superhuman abilities, ...",a miraculous solution that includes a set of w...,-0.81


In [220]:
df.to_csv(output_path+"results.csv", index=False)

In [221]:
positive_summary = re.sub(r"Letter from [^:]+:\s*", "", summary_results_pos[0]["summary_text"])
negative_summary = re.sub(r"Letter from [^:]+:\s*", "", summary_results_neg[0]["summary_text"])

In [222]:
plt.figure()
plt.scatter(df.sort_values("polarity")["polarity"], df.sort_values("polarity")["name"])
plt.xlabel("Sentiment Score")
plt.ylabel("Customer")
plt.title("Sentiment Scores by Letter")
plt.tight_layout()
plt.savefig(output_path+"polarity_scatter.png")
plt.close()

In [229]:
neg_name = most_negative["name"].iloc[0]
pos_name = most_positive["name"].iloc[0]

neg_row = df[df["name"] == neg_name].iloc[0]
pos_row = df[df["name"] == pos_name].iloc[0]

negative_df = df[df["sentiment"] == "negative"][["name", "summary", "request"]].copy()
positive_df = df[df["sentiment"] == "positive"][["name", "summary", "request"]].copy()

# Optional: shorten long text for cleaner tables
for table_df in [negative_df, positive_df]:
    table_df["summary"] = table_df["summary"].astype(str).str.slice(0, 120)
    table_df["request"] = table_df["request"].astype(str).str.slice(0, 60)

doc = Document()

# Margins
section = doc.sections[0]
section.top_margin = Inches(0.75)
section.bottom_margin = Inches(0.75)
section.left_margin = Inches(0.85)
section.right_margin = Inches(0.85)

# Base style
styles = doc.styles
styles["Normal"].font.name = "Aptos"
styles["Normal"].font.size = Pt(10.5)

# Logo
logo_path = output_path + "logo.png"  # change this if needed
add_logo_to_header(doc, logo_path, width_inches=1.0)

# Title page section
add_report_title(doc, "Blue Cow Customer Feedback Report")
add_subtitle(doc, "Sentiment, summarisation and customer-request analysis of generated feedback letters")

# Introduction
add_heading_custom(doc, "1. Introduction", level=1)
add_body_text(
    doc,
    "This report summarises the results of sentiment analysis, summarisation, and "
    "question answering applied to a set of customer feedback letters generated and "
    "analysed using Hugging Face models."
)

# Summary of findings
add_heading_custom(doc, "2. Summary of Findings", level=1)

add_body_text(doc, f"Number of letters analysed: {len(df)}")
add_body_text(
    doc,
    f"Negative: {len(df[df['sentiment'] == 'negative'])}. "
    f"Positive: {len(df[df['sentiment'] == 'positive'])}. "
    f"Neutral: {len(df[df['sentiment'] == 'neutral'])}."
)

doc.add_picture(output_path+"polarity_scatter.png", width=Inches(5.7))
add_caption(doc, "Figure 1. Signed sentiment scores across customer letters.")

add_heading_custom(doc, "Summary of positive feedback", level=3)
add_body_text(doc, positive_summary)

add_heading_custom(doc, "Summary of negative feedback", level=3)
add_body_text(doc, negative_summary)

# Negative feedback
add_heading_custom(doc, "3. Negative Feedback", level=1)
add_df_to_doc_styled(doc, negative_df, title="Letter analysis results")

add_heading_custom(doc, "3.1 Case Study: Most Extreme Negative Feedback", level=2)
add_textbox_block(doc, "Customer letter", letters[neg_name], fill="F7F8FC", border=GG_BLUE)

add_heading_custom(doc, "Summary of complaint", level=3)
add_body_text(doc, neg_row["summary"])

add_heading_custom(doc, "Customer request", level=3)
add_body_text(doc, neg_row["request"])

add_heading_custom(doc, "Suggested response", level=3)
add_textbox_block(doc, "Proposed company response", response_neg, fill="F7F8FC", border=GG_BLUE)

# Positive feedback
add_heading_custom(doc, "4. Positive Feedback", level=1)
add_df_to_doc_styled(doc, positive_df, title="Letter analysis results")

add_heading_custom(doc, "4.1 Case Study: Most Extreme Positive Feedback", level=2)
add_textbox_block(doc, "Customer letter", letters[pos_name], fill="F7F8FC", border=GG_BLUE)

add_heading_custom(doc, "Summary of praise", level=3)
add_body_text(doc, pos_row["summary"])

add_heading_custom(doc, "Customer request", level=3)
add_body_text(doc, pos_row["request"])

add_heading_custom(doc, "Suggested response", level=3)
add_textbox_block(doc, "Proposed company response", response_pos, fill="F7F8FC", border=GG_BLUE)

In [230]:
doc.save(output_path + "report.docx")